In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LinearRegression


# ============================================================
# CONFIG
# ============================================================

INPUT_CSV = "region_avg.csv"
OUTPUT_CSV = "region_temp_extended.csv"

DATE_COL = "date"
REGION_COL = "region_code"
TEMP_COL = "daily_avg_temperature"

REMOVE_LEAP_DAY_FOR_ML = True
MIN_HISTORY_ROWS = 5 * 365   # keep enough history for year-based features

# lag/difference structure from notebook / paper logic
LAGS = {
    "next_day": -1,
    "yday": 1,
    "IIdays_ago": 2,
    "IIIdays_ago": 3,
    "IVdays_ago": 4,
    "Vdays_ago": 5,
    "VIdays_ago": 6,
    "VIIdays_ago": 7,
    "last_year": 365,
    "last_2year": 365 * 2,
    "last_3year": 365 * 3,
    "last_4year": 365 * 4,
    "last_5year": 365 * 5,
    "Imonth_ago": 30,
    "h1_last_year": 364,
    "h1_last_2years": 364 + 365,
    "h1_last_3years": 364 + (365 * 2),
    "h1_last_4years": 364 + (365 * 3),
    "h1_last_5years": 364 + (365 * 4),
    "h1_ly_2days": (365 - 1) + 2,
    "h1_ly_3day": (365 - 1) + 3,
    "h1_ly_4day": (365 - 1) + 4,
    "h1_ly_5day": (365 - 1) + 5,
    "h1_ly_6day": (365 - 1) + 6,
    "h1_ly_7day": (365 - 1) + 7,
    "h1_ly_next_1day": (365 - 1) - 1,
    "h1_ly_next_2days": (365 - 1) - 2,
    "h1_ly_next_3days": (365 - 1) - 3,
    "h1_ly_next_4days": (365 - 1) - 4,
    "h1_ly_next_5days": (365 - 1) - 5,
    "h1_ly_next_6days": (365 - 1) - 6,
    "h1_ly_next_7days": (365 - 1) - 7,
}

DIFFS = {
    "diff_1year": 365,
    "diff_2year": 365 * 2,
    "diff_3year": 365 * 3,
    "diff_4year": 365 * 4,
    "diff_5year": 365 * 5,
    "diff_yday": 1,
    "diff_2days": 2,
    "diff_3days": 3,
    "diff_4days": 4,
    "diff_5days": 5,
    "diff_6days": 6,
    "diff_7days": 7,
}


# ============================================================
# HELPERS
# ============================================================

def add_calendar_fields(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["dayofyear"] = df[DATE_COL].dt.dayofyear

    leap_after_feb = (df[DATE_COL].dt.is_leap_year) & (df[DATE_COL].dt.month >= 3)
    df.loc[leap_after_feb, "dayofyear"] -= 1

    feb29 = (df[DATE_COL].dt.month == 2) & (df[DATE_COL].dt.day == 29)
    df.loc[feb29, "dayofyear"] = 366

    df["weeknum"] = df[DATE_COL].dt.isocalendar().week.astype(int)
    df["month"] = df[DATE_COL].dt.month.astype(int)
    return df


def reindex_daily(group: pd.DataFrame) -> pd.DataFrame:
    g = group.copy().sort_values(DATE_COL)
    full_dates = pd.date_range(g[DATE_COL].min(), g[DATE_COL].max(), freq="D")
    g = g.set_index(DATE_COL).reindex(full_dates).rename_axis(DATE_COL).reset_index()
    g[REGION_COL] = group[REGION_COL].iloc[0]
    return g


def impute_temperature(group: pd.DataFrame) -> pd.DataFrame:
    """
    Adapted from the notebook logic:
    TAVG_imptd = average of:
      - day-of-year average
      - past 7-day mean
      - next 7-day mean
    implemented as average of day-of-year average and average(past7, next7)
    """
    g = reindex_daily(group)
    g = add_calendar_fields(g)

    g[TEMP_COL] = pd.to_numeric(g[TEMP_COL], errors="coerce")

    # past 7 day average, excluding current day
    g["past7days_avgT"] = g[TEMP_COL].shift(1).rolling(window=7, min_periods=2).mean()

    # next 7 day average, excluding current day
    rev = g.iloc[::-1].copy()
    rev["next7days_avgT"] = rev[TEMP_COL].shift(1).rolling(window=7, min_periods=2).mean()
    g["next7days_avgT"] = rev["next7days_avgT"].iloc[::-1].values

    # day-of-year average using available values
    doy_avg = g.groupby("dayofyear")[TEMP_COL].mean().round(3)
    g["doyavg"] = g["dayofyear"].map(doy_avg)

    g["surround_avg"] = g[["past7days_avgT", "next7days_avgT"]].mean(axis=1)

    g["TAVG_imptd"] = g[TEMP_COL]
    miss = g["TAVG_imptd"].isna()

    g.loc[miss, "TAVG_imptd"] = g.loc[miss, ["doyavg", "surround_avg"]].mean(axis=1)

    # fallbacks
    g["TAVG_imptd"] = g["TAVG_imptd"].interpolate(method="linear", limit_direction="both")
    g["TAVG_imptd"] = g["TAVG_imptd"].fillna(g["doyavg"])
    g["TAVG_imptd"] = g["TAVG_imptd"].fillna(g["TAVG_imptd"].mean())

    return g


def drop_leap_day_for_ml(df: pd.DataFrame) -> pd.DataFrame:
    if not REMOVE_LEAP_DAY_FOR_ML:
        return df.copy()
    return df.loc[~((df[DATE_COL].dt.month == 2) & (df[DATE_COL].dt.day == 29))].copy()


def add_raw_training_helpers(group: pd.DataFrame) -> pd.DataFrame:
    g = group.copy().sort_values(DATE_COL)

    g["rawtempdoyavg"] = g.groupby("dayofyear")["TAVG_imptd"].transform("mean")
    g["pdtn_doy"] = np.where(g["dayofyear"] == 365, 1, g["dayofyear"] + 1)

    pred_doy_avg = (
        g.groupby("dayofyear")["rawtempdoyavg"]
        .mean()
        .rename("rawtemppdtndoyavg")
        .reset_index()
    )

    g = g.merge(pred_doy_avg, how="left", left_on="pdtn_doy", right_on="dayofyear", suffixes=("", "_y"))
    g = g.drop(columns=["dayofyear_y"])
    return g


def add_trend_and_seasonality(group: pd.DataFrame) -> pd.DataFrame:
    g = group.copy().sort_values(DATE_COL).reset_index(drop=True)

    X = np.arange(len(g)).reshape(-1, 1)
    y = g["TAVG_imptd"].values

    linreg = LinearRegression(fit_intercept=True)
    linreg.fit(X, y)
    trend = linreg.predict(X)

    g["trend"] = trend
    g["detrended"] = y - trend

    # notebook uses FFT filtering around dominant frequency
    data = g["TAVG_imptd"].values
    fft = np.fft.fft(data)
    power_spectrum = np.abs(fft) ** 2
    frequencies = np.fft.fftfreq(len(data))

    peak_frequency = frequencies[np.argmax(power_spectrum[1:]) + 1] if len(data) > 1 else 0.0
    if peak_frequency == 0:
        filtered_data = np.zeros_like(data, dtype=float)
    else:
        mask = (np.abs(frequencies) > peak_frequency - 0.01) & (np.abs(frequencies) < peak_frequency + 0.01)
        filtered_fft = fft.copy()
        filtered_fft[~mask] = 0
        filtered_data = np.fft.ifft(filtered_fft)

    g["four_seas"] = np.real(filtered_data)
    g["de_trend_seas"] = g["detrended"] - g["four_seas"]

    return g


def add_deseasonalized_stats(group: pd.DataFrame) -> pd.DataFrame:
    g = group.copy().sort_values(DATE_COL)

    doy_mean = g.groupby("dayofyear")["de_trend_seas"].transform("mean")
    doy_var = g.groupby("dayofyear")["de_trend_seas"].transform("var")
    doy_std = g.groupby("dayofyear")["de_trend_seas"].transform("std")

    g["dts_doyavge"] = doy_mean
    g["dts_doyvar"] = doy_var.fillna(0.0)
    g["dts_doystdv"] = doy_std.fillna(0.0)

    return g


def add_lag_and_diff_features(group: pd.DataFrame) -> pd.DataFrame:
    g = group.copy().sort_values(DATE_COL)

    for key, value in LAGS.items():
        g[key] = g["de_trend_seas"].shift(value)

    for key, value in DIFFS.items():
        g[key] = g["de_trend_seas"].diff(value)

    g["last_7_1day_deltas_mean"] = g["diff_yday"].rolling(window=7).mean()
    g["last_7_1day_deltas_min"] = g["diff_yday"].rolling(window=7).min()
    g["last_7_1day_deltas_max"] = g["diff_yday"].rolling(window=7).max()

    return g


def process_region(group: pd.DataFrame) -> pd.DataFrame:
    g = group.copy()
    g[DATE_COL] = pd.to_datetime(g[DATE_COL])

    g = impute_temperature(g)
    g = drop_leap_day_for_ml(g)
    g = add_calendar_fields(g)
    g = add_raw_training_helpers(g)
    g = add_trend_and_seasonality(g)
    g = add_deseasonalized_stats(g)
    g = add_lag_and_diff_features(g)

    # notebook uses "name"
    g["name"] = g[REGION_COL]
    g["station_code"] = g[REGION_COL]

    # keep a clean order
    first_cols = [
        DATE_COL,
        "station_code",
        "name",
        REGION_COL,
        "dayofyear",
        "weeknum",
        "month",
        TEMP_COL,
        "TAVG_imptd",
        "doyavg",
        "rawtempdoyavg",
        "pdtn_doy",
        "rawtemppdtndoyavg",
        "trend",
        "detrended",
        "four_seas",
        "de_trend_seas",
        "dts_doyavge",
        "dts_doyvar",
        "dts_doystdv",
    ]
    first_cols = [c for c in first_cols if c in g.columns]
    other_cols = [c for c in g.columns if c not in first_cols]
    g = g[first_cols + other_cols]

    # drop rows with insufficient lag history
    g = g.iloc[MIN_HISTORY_ROWS:].copy()

    return g


# ============================================================
# MAIN
# ============================================================

def main():
    input_path = Path(INPUT_CSV)
    output_path = Path(OUTPUT_CSV)

    df = pd.read_csv(input_path)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])

    required = {DATE_COL, REGION_COL, TEMP_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df[[DATE_COL, REGION_COL, TEMP_COL]].copy()
    df = df.sort_values([REGION_COL, DATE_COL]).reset_index(drop=True)

    out = []
    for region, g in df.groupby(REGION_COL):
        print(f"Processing region {region} ...")
        out.append(process_region(g))

    final_df = pd.concat(out, ignore_index=True)
    final_df = final_df.sort_values([REGION_COL, DATE_COL]).reset_index(drop=True)

    final_df.to_csv(output_path, index=False)

    print("\nDone.")
    print(f"Saved to: {output_path}")
    print(f"Shape: {final_df.shape}")
    print("\nCore columns:")
    core = [
        "name", "TAVG_imptd", "trend", "detrended", "four_seas",
        "de_trend_seas", "dayofyear", "month", "dts_doyavge", "dts_doyvar"
    ]
    print([c for c in core if c in final_df.columns])


if __name__ == "__main__":
    main()

Processing region 11 ...
Processing region 24 ...
Processing region 27 ...
Processing region 28 ...
Processing region 32 ...
Processing region 44 ...
Processing region 52 ...
Processing region 53 ...

Done.
Saved to: region_temp_extended.csv
Shape: (143810, 70)

Core columns:
['name', 'TAVG_imptd', 'trend', 'detrended', 'four_seas', 'de_trend_seas', 'dayofyear', 'month', 'dts_doyavge', 'dts_doyvar']
